# Data Preparation Part 1 Practical Session

Student version for a 90-minute classroom practical built around the Ames Housing dataset.

Work through the TODO cells, write down the requested numeric answers, and keep the intermediate data objects because later tasks reuse them.

## Learning Goals
- quantify missingness and reason about likely missing-data mechanisms
- measure the impact of median imputation and log transforms
- compare univariate and multivariate outlier detection ideas
- practice ordinal encoding, one-hot reasoning, binning, and robust scaling
- assemble a reproducible preprocessing pipeline and evaluate a simple regression model


## 1. Setup and Data Access (10 min)

Start by importing dependencies and loading the Ames Housing dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Libraries loaded successfully!")


## Task 1: Load and Inspect

**Instructions:**
Load the Ames Housing dataset into a DataFrame named `df`.
- URL: `https://drive.google.com/uc?export=download&id=11m25c8jLsqHV6pePePACd_ntPx9Wz7Pc`
- Remember: missing values in this file are marked `"NA"`. Make sure pandas parses them as NaNs (`na_values="NA"`).

**Calculation 1:** What is the exact number of rows and columns in the raw dataset?


In [ ]:
# TODO:
# 1. Read the CSV from the provided URL.
# 2. Parse both "NA" and empty strings as missing values.
# 3. Use the Order column as the index.
# 4. Print the shape and display the first rows.

# df = ...


**Answer 1:** [Student provides rows/columns].


---
# Part 2: Missing Values Impact (15 min)


## Task 2: Quantify Missingness
**Calculation 2:** What is the exact number of missing values in the `Lot Frontage` feature? And what missingness mechanism (MCAR, MAR, MNAR) does this specific feature likely represent?


In [ ]:
# TODO:
# 1. Count missing values in df['Lot Frontage'].
# 2. Print the count.
# 3. In the answer cell, argue which missingness mechanism seems most plausible.


**Answer 2:** [Number] missing values. Likely mechanism: [MCAR / MAR / MNAR + one-sentence justification].


## Task 3: The Impact of Imputation
If you apply Median Imputation to `Lot Frontage`, it changes the distribution. Let's quantify how much.

**Calculation 3:** Create a copy of the dataframe `df_work = df.copy()`. Impute `Lot Frontage` using its training set median. What is the EXACT new overall mean of `Lot Frontage` across all 2930 rows after imputation (rounded to 2 decimal places)?


In [ ]:
# TODO:
# 1. Create df_work = df.copy().
# 2. Compute the median of Lot Frontage.
# 3. Create Lot Frontage_imputed with fillna(median).
# 4. Compute and print the new overall mean rounded to 2 decimals.


**Answer 3:** New mean after imputation is [Value].


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_work['Lot Frontage'].dropna(), kde=True, label='Original', alpha=0.45)
sns.histplot(df_work['Lot Frontage_imputed'], kde=True, label='Median-imputed', alpha=0.45)
plt.title('Lot Frontage: original vs median-imputed')
plt.xlabel('Lot Frontage (feet)')
plt.ylabel('Frequency')
plt.legend()
plt.show()


---
## 2. Outliers and Transformations (20 min)

## Task 4: IQR Thresholds
The `SalePrice` feature in real estate is famously right-skewed.

**Calculation 4:** Calculate the Interquartile Range (IQR) for `SalePrice`. Based on the Tukey rule ($Q3 + 1.5 \times IQR$), what is the exact mathematical maximum boundary for `SalePrice` before a house is deemed an outlier? How many houses in the dataset exceed this limit?


In [ ]:
# TODO:
# 1. Compute Q1, Q3, and the IQR for SalePrice.
# 2. Compute the Tukey upper bound: Q3 + 1.5 * IQR.
# 3. Count how many houses exceed that boundary.
# 4. Print the boundary and the count.


**Answer 4:** Upper boundary is [Amount]. Outlier count is [Count].


## Task 5: Stabilizing Skewness
Because of these massive multi-million dollar outliers, we want to apply a log transformation.

**Calculation 5:** Apply the transformation `np.log1p()` to the `SalePrice` column. After applying this transformation, what is the exact maximum value of this column rounded to 2 decimal places?


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_work['SalePrice'], kde=True)
plt.title('Distribution of SalePrice')
plt.xlabel('SalePrice')
plt.ylabel('Frequency')
plt.show()

skewness = df_work['SalePrice'].skew()
kurtosis = df_work['SalePrice'].kurt()
print(f"Skewness of SalePrice: {skewness:.2f}")
print(f"Kurtosis of SalePrice: {kurtosis:.2f}")

### Explanation of Skewness and Kurtosis for SalePrice

*   **Skewness (1.74):** This value indicates the asymmetry of the probability distribution of `SalePrice`. A positive skewness (like 1.74) means that the tail of the distribution is longer on the right side, and the majority of the data points (housing prices) are concentrated on the lower end, with a few very high prices pulling the mean to the right. This is common for real estate prices.

*   **Kurtosis (5.12):** This measures the 'tailedness' of the distribution. A kurtosis value greater than 3 (which 5.12 is) indicates a 'leptokurtic' distribution. This means the distribution has heavier tails and a sharper peak compared to a normal distribution. In practical terms, for `SalePrice`, it suggests there are more extreme outliers (both very low and very high prices) than you would expect from a normal distribution, and the prices are more concentrated around the mean.

In [ ]:
# TODO:
# 1. Create a log-transformed SalePrice column with np.log1p.
# 2. Compute the maximum transformed value.
# 3. Print it rounded to 2 decimals.

# df_work['SalePrice_Log'] = ...


**Answer 5:** Maximum logged value is [Value].


## Task 6: Multivariate Outliers (Isolation Forest)
Sometimes univariate IQR isn't enough. Let's look for outliers in a multi-dimensional space.

**Calculation 6:** Train an `IsolationForest` (with `contamination=0.01` and `random_state=42`) exclusively on a subset containing only `Lot Area` and `SalePrice`. Drop any nulls from this subset before fitting. How many anomalies (indicated by -1) does the forest detect?


In [ ]:
# TODO:
# 1. Fit IsolationForest(contamination=0.01, random_state=42) on ['Lot Area', 'SalePrice'].
# 2. Drop missing rows before fitting.
# 3. Count how many predictions equal -1.
# 4. Print the anomaly count.


**Answer 6:** Isolation Forest detected [Value] anomalies.


---
## 3. Encodings and Binning (20 min)

## Task 7: Ordinal Encoding
The feature `Exter Qual` contains ordinal text data: 'Ex' (Excellent), 'Gd' (Good), 'TA' (Typical/Average), 'Fa' (Fair), 'Po' (Poor).

**Calculation 7:** Map these string categories to integers dictionary-style: {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}. Apply this to the `Exter Qual` column. What is the newly calculated mean of this numerical column (rounded to 2 decimal places)?


In [ ]:
# TODO:
# 1. Map Exter Qual with {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}.
# 2. Store the encoded values in a new column.
# 3. Compute and print the mean rounded to 2 decimals.


**Answer 7:** Mean encoded quality is [Value].


## Task 8: Data Binning (Discretization)
Continuous features like `Year Built` can sometimes be too noisy. We can group them.

**Calculation 8:** Discretize `Year Built` into exactly 5 equal-sized (quantile) bins using `pd.qcut()`. Look at the value counts of the resulting bins. How many houses fall into the absolute newest age bin?


In [ ]:
# TODO:
# 1. Use pd.qcut on Year Built with q=5.
# 2. Inspect the value counts of the bins.
# 3. Find how many houses fall into the newest bin.
# 4. Print the counts and the final number.


**Answer 8:** [Count] houses fall into the newest bin.


## Task 9: One-Hot Categorical Count
Look at the nominal `Neighborhood` column.

**Calculation 9:** How many unique neighborhoods are represented in the dataset? If you were to pass `Neighborhood` directly into a `OneHotEncoder(drop='first', sparse_output=False)`, exactly how many NEW boolean/numeric columns would be added to your dataset representing this single feature?


In [ ]:
# TODO:
# 1. Count the number of unique Neighborhood values.
# 2. Recall that OneHotEncoder(drop='first') removes one dummy column.
# 3. Compute how many new columns would be added.
# 4. Print both numbers.


**Answer 9:** [X] unique neighborhoods and [Y] added one-hot columns.


---
## 4. Feature Engineering, Scaling, and Pipelines (25 min)

## Task 10: Total Square Footage Engineering
House price is highly dependent on total size.

**Calculation 10:** Create a new feature `Total_Square_Footage` by summing `Total Bsmt SF`, `1st Flr SF`, and `2nd Flr SF` together (handle NA values safely by filling with 0 before summing). What is the absolute highest total combined square footage of any house in this dataset?


In [ ]:
# TODO:
# 1. Create Total_Square_Footage from basement, first-floor, and second-floor area.
# 2. Fill missing values with 0 before summing.
# 3. Compute the maximum total square footage.
# 4. Print the largest value.


**Answer 10:** Largest total square footage is [Value] sqft.


## Task 11: Robust Scaling the Engine
If we are using distance-based algorithms, scaling `Total_Square_Footage` is mandatory.

**Calculation 11:** Apply the `RobustScaler` (which uses IQR) to your `Total_Square_Footage` column. After scaling, what is the newly scaled value of that massive outlier from Task 10 (the largest house)? Round to 2 decimal places.


In [ ]:
# TODO:
# 1. Fit RobustScaler on Total_Square_Footage.
# 2. Store the scaled values in a new column.
# 3. Find the scaled value of the largest house from Task 10.
# 4. Print it rounded to 2 decimals.


**Answer 11:** Scaled value of the largest house is [Value].


## Task 12: The Final Pipeline Blueprint
Put it all together mapping pipelines into a ColumnTransformer!

1. Select these 4 numeric features: `['Lot Frontage', 'Total Bsmt SF', '1st Flr SF', 'Gr Liv Area']`
2. Apply: Median Imputation -> StandardScaler
3. Select this 1 categorical feature: `['Neighborhood']`
4. Apply: Most Frequent Imputer -> OneHotEncoder(drop='first', sparse_output=False)

**Calculation 12:** If you fit/transform this exact `ColumnTransformer` (named `preprocess`) on the training set (e.g. `X.shape[0]` rows), what is the exact `shape` (rows, columns) of the resulting numpy array?


In [ ]:
# TODO:
# 1. Define numeric and categorical feature lists exactly as requested above.
# 2. Build the numeric and categorical pipelines.
# 3. Combine them in a ColumnTransformer named preprocess.
# 4. Fit-transform the selected subset and print the resulting shape.


**Answer 12:** Resulting transformed shape is (rows, [cols]).


## Task 13: Model Evaluation Challenge
Finally, append a `Ridge(alpha=10.0)` model to your `ColumnTransformer` from Task 12 inside a final Pipeline. We're going to predict `SalePrice`.

**Calculation 13:**
1. `train_test_split` your data (80% train, 20% test, `random_state=42`).
2. Train the pipeline on `X_train`, `y_train`.
3. Predict on `X_test`.
What is the Root Mean Squared Error (RMSE) on the test set, rounded to the nearest integer?


In [ ]:
# TODO:
# 1. Split X and y with train_test_split(test_size=0.2, random_state=42).
# 2. Build a Pipeline with preprocess and Ridge(alpha=10.0).
# 3. Fit on the training data and predict on the test data.
# 4. Compute RMSE and R2, then print both.


**Answer 13:** Test RMSE is [Integer].
